<a href="https://colab.research.google.com/github/Ena-AlexBrush/Fine-Tuning-Experiments/blob/main/GRPO_w_key_equal_prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install trl[GRPOTrainer]
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 56.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
from trl import GRPOTrainer, GRPOConfig
from datasets import load_dataset
import re

In [11]:

# 1. Load your dataset
train_dataset = load_dataset("hendzh/PromptShield", split="train[:50]")
eval_dataset = load_dataset("hendzh/PromptShield", split="validation[:50]")

# 2. Define a simple reward function
def reward_func(completions, **kwargs):
    """Example: Reward longer completions"""
    return [float(len(completion)) for completion in completions]

README.md:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

train.json: reconstructing file:   0%|          |  0.00B / 12.2MB            

train.json: downloading bytes:           |  0.00B            

validation.json:   0%|          | 0.00/646k [00:00<?, ?B/s]

test.json: reconstructing file:   0%|          |  0.00B / 18.3MB            

test.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18909 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/23516 [00:00<?, ? examples/s]

In [13]:

# 3. Configure training
training_args = GRPOConfig(
    output_dir="output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    logging_steps=10,
    num_generations=4,
)

# 4. Initialize and train
trainer = GRPOTrainer(
    model="HuggingFaceTB/SmolLM2-135M",
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    reward_funcs=reward_func,
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [6]:
print(train_dataset[0].keys())

dict_keys(['prompt', 'quality', 'metadata', 'avg_rating', 'num_responses', 'agreement_ratio', 'raw_responses', 'kind', 'cluster_description', 'topic'])


In [14]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,-0.402160
20,-0.171863
30,-0.165993
40,-0.380861
50,-0.294567
60,-0.245183
70,-0.426580
80,-0.278121
90,-0.240848
100,-0.264765


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=150, training_loss=-0.2715542928377787, metrics={'train_runtime': 2162.2326, 'train_samples_per_second': 0.069, 'train_steps_per_second': 0.069, 'total_flos': 0.0, 'train_loss': -0.2715542928377787, 'epoch': 3.0})

In [15]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Num Tokens,Completions/mean Length,Completions/min Length,Completions/max Length,Completions/clipped Ratio,Completions/mean Terminated Length,Completions/min Terminated Length,Completions/max Terminated Length,Rewards/reward Func/mean,Rewards/reward Func/std,Reward,Reward Std,Frac Reward Zero Std,Entropy,Clip Ratio/low Mean,Clip Ratio/high Mean,Clip Ratio/region Mean,Clip Ratio/low Min,Clip Ratio/high Max
-0.291556,-0.306932,150,223532.000000,203.485000,63.080000,256.000000,0.690000,86.636000,52.840000,129.720000,835.055000,377.561196,835.055000,377.561196,0.000000,3.624184,0.000000,0.000000,0.000000,0.000000,0.000000


{'eval_loss': -0.3069320321083069,
 'eval_num_tokens': 223532.0,
 'eval_completions/mean_length': 203.485,
 'eval_completions/min_length': 63.08,
 'eval_completions/max_length': 256.0,
 'eval_completions/clipped_ratio': 0.69,
 'eval_completions/mean_terminated_length': 86.63600006103516,
 'eval_completions/min_terminated_length': 52.84,
 'eval_completions/max_terminated_length': 129.72,
 'eval_rewards/reward_func/mean': 835.055,
 'eval_rewards/reward_func/std': 377.5611962890625,
 'eval_reward': 835.055,
 'eval_reward_std': 377.5611962890625,
 'eval_frac_reward_zero_std': 0.0,
 'eval_entropy': 3.624184331893921,
 'eval_clip_ratio/low_mean': 0.0,
 'eval_clip_ratio/high_mean': 0.0,
 'eval_clip_ratio/region_mean': 0.0,
 'eval_clip_ratio/low_min': 0.0,
 'eval_clip_ratio/high_max': 0.0}